In [1]:
# Compare groups containing 'vicreg' and 'linear': mean test F1 and p-value
import pandas as pd
import numpy as np
import wandb
from typing import Any, Dict

# Optional permutation test; fallback to Welch's t-test if unavailable
try:
    from mlxtend.evaluate import permutation_test
except Exception:
    permutation_test = None
from scipy.stats import ttest_ind

api = wandb.Api()
project_path = "francisco-soto-u-universidad-de-chile/Paper-Periodic-LightCurves-Classification"

# Fetch finished runs (keep API-side filtering minimal, filter groups in Python)
try:
    runs = api.runs(project_path, filters={"state": "finished"})
except Exception as e:
    print(f"W&B API error: {e}. Ensure WANDB_API_KEY is set and you have access.")
    runs = []

rows = []
for run in runs:
    grp = getattr(run, "group", None)
    if not isinstance(grp, str):
        continue
    grp_low = grp.lower()
    if ("vicreg" not in grp_low) and ("linear" not in grp_low):
        continue
    summary = run.summary._json_dict
    test_f1 = summary.get("test/f1")
    if isinstance(test_f1, (int, float)):
        rows.append({
            "run_name": run.name,
            "group": grp,
            "supergroup": "vicreg" if "vicreg" in grp_low else ("linear" if "linear" in grp_low else "other"),
            "test_f1": float(test_f1),
        })

if not rows:
    print("No finished runs found with groups containing 'vicreg' or 'linear' and a numeric test/f1 in summary.")
else:
    df = pd.DataFrame(rows)
    # Keep only the two supergroups of interest
    df = df[df["supergroup"].isin(["vicreg", "linear"])]

    # Aggregate stats
    grouped = (df.groupby(["supergroup"])  
                 .agg(mean_test_f1=("test_f1", "mean"),
                      std_test_f1=("test_f1", "std"),
                      n=("test_f1", "count"))
                 .reset_index())
    display(grouped)

    # Prepare arrays for significance test
    arr_v = df[df["supergroup"] == "vicreg"]["test_f1"].to_numpy()
    arr_l = df[df["supergroup"] == "linear"]["test_f1"].to_numpy()

    p_perm = None
    if permutation_test is not None and len(arr_v) > 0 and len(arr_l) > 0:
        try:
            p_perm = float(permutation_test(arr_v, arr_l, func='x_mean > y_mean', method='approximate', num_rounds=10000, paired=False))
        except Exception:
            p_perm = None

    # Fallback to Welch's t-test (two-sided); also report one-sided if desired
    p_ttest = None
    one_sided = None
    if len(arr_v) > 1 and len(arr_l) > 1:
        try:
            tstat, p_two = ttest_ind(arr_v, arr_l, equal_var=False, nan_policy='omit')
            p_ttest = float(p_two)
            # Optional one-sided p-value (vicreg > linear)
            one_sided = float(p_two / 2) if (np.mean(arr_v) > np.mean(arr_l)) else float(1 - p_two / 2)
        except Exception:
            pass

    print("\nSignificance between vicreg and linear (test/f1):")
    if p_perm is not None:
        print(f"Permutation test (vicreg > linear) p-value: {p_perm:.6f}")
    else:
        print("Permutation test unavailable or failed.")
    if p_ttest is not None:
        print(f"Welch t-test (two-sided) p-value: {p_ttest:.6f}")
        if one_sided is not None:
            print(f"Welch t-test (one-sided, vicreg > linear) p-value: {one_sided:.6f}")

    # Save detailed rows and summary
    try:
        out_dir = Path("paper_notebooks/csv")
        out_dir.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_dir / "vicreg_linear_runs_test_f1.csv", index=False)
        grouped.to_csv(out_dir / "vicreg_linear_summary_test_f1.csv", index=False)
        print("\nSaved:\n - paper_notebooks/csv/vicreg_linear_runs_test_f1.csv\n - paper_notebooks/csv/vicreg_linear_summary_test_f1.csv")
    except Exception as e:
        print(f"Failed to save CSVs: {e}")

,supergroup,mean_test_f1,std_test_f1,n
0,vicreg,0.554585,0.095938,100



Significance between vicreg and linear (test/f1):
Permutation test unavailable or failed.
Failed to save CSVs: name 'Path' is not defined


In [2]:
# VICREG subgroups: separate four groups, mean test F1 and pairwise p-values
import pandas as pd
import numpy as np
import wandb
from typing import Any, Dict

# Optional permutation test; fallback to Welch's t-test if unavailable
try:
    from mlxtend.evaluate import permutation_test
except Exception:
    permutation_test = None
from scipy.stats import ttest_ind

api = wandb.Api()
project_path = "francisco-soto-u-universidad-de-chile/Paper-Periodic-LightCurves-Classification"

# Fetch finished runs and keep only groups containing 'vicreg'
try:
    runs = api.runs(project_path, filters={"state": "finished"})
except Exception as e:
    print(f"W&B API error: {e}. Ensure WANDB_API_KEY is set and you have access.")
    runs = []

rows = []
for run in runs:
    grp = getattr(run, "group", None)
    if not isinstance(grp, str):
        continue
    grp_low = grp.lower()
    if "vicreg" not in grp_low:
        continue
    summary = run.summary._json_dict
    test_f1 = summary.get("test/f1")
    if isinstance(test_f1, (int, float)):
        rows.append({
            "run_name": run.name,
            "group": grp,
            "test_f1": float(test_f1),
        })

if not rows:
    print("No finished runs found with groups containing 'vicreg' and a numeric test/f1 in summary.")
else:
    df = pd.DataFrame(rows)
    vic_groups = sorted(df["group"].unique())
    print(f"Found VICREG groups (n={len(vic_groups)}):")
    for g in vic_groups:
        print(f" - {g} (runs={len(df[df['group']==g])})")
    if len(vic_groups) != 4:
        print("Warning: Expected 4 VICREG groups. Proceeding with available groups.")

    # Per-group stats
    grouped = (df.groupby(["group"])  
                 .agg(mean_test_f1=("test_f1", "mean"),
                      std_test_f1=("test_f1", "std"),
                      n=("test_f1", "count"))
                 .reset_index())
    display(grouped.sort_values("group"))

    # Pairwise p-values between VICREG groups
    results = []
    for i in range(len(vic_groups)):
        for j in range(i+1, len(vic_groups)):
            g1, g2 = vic_groups[i], vic_groups[j]
            arr1 = df[df["group"] == g1]["test_f1"].to_numpy()
            arr2 = df[df["group"] == g2]["test_f1"].to_numpy()
            p_perm = None
            if permutation_test is not None and len(arr1) > 0 and len(arr2) > 0:
                try:
                    p_perm = float(permutation_test(arr1, arr2, func='x_mean > y_mean', method='approximate', num_rounds=10000, paired=False))
                except Exception:
                    p_perm = None
            p_ttest = None
            one_sided = None
            if len(arr1) > 1 and len(arr2) > 1:
                try:
                    tstat, p_two = ttest_ind(arr1, arr2, equal_var=False, nan_policy='omit')
                    p_ttest = float(p_two)
                    # One-sided p-value (mean g1 > mean g2)
                    one_sided = float(p_two / 2) if (np.mean(arr1) > np.mean(arr2)) else float(1 - p_two / 2)
                except Exception:
                    pass
            results.append({
                "group_a": g1,
                "mean_a": float(np.mean(arr1)) if len(arr1) else np.nan,
                "n_a": int(len(arr1)),
                "group_b": g2,
                "mean_b": float(np.mean(arr2)) if len(arr2) else np.nan,
                "n_b": int(len(arr2)),
                "p_perm_one_sided_a_gt_b": p_perm,
                "p_ttest_two_sided": p_ttest,
                "p_ttest_one_sided_a_gt_b": one_sided,
            })

    res_df = pd.DataFrame(results)
    if res_df.empty:
        print("No pairwise results to show.")
    else:
        display(res_df.sort_values(["group_a", "group_b"]))
        # Save CSVs
        try:
            from pathlib import Path
            out_dir = Path("paper_notebooks/csv")
            out_dir.mkdir(parents=True, exist_ok=True)
            grouped.to_csv(out_dir / "vicreg_groups_test_f1_summary.csv", index=False)
            res_df.to_csv(out_dir / "vicreg_groups_pairwise_pvals.csv", index=False)
            print("\nSaved:\n - paper_notebooks/csv/vicreg_groups_test_f1_summary.csv\n - paper_notebooks/csv/vicreg_groups_pairwise_pvals.csv")
        except Exception as e:
            print(f"Failed to save CSVs: {e}")


Found VICREG groups (n=4):
 - VICReg-Linear-Aug-RN (runs=25)
 - VICReg-Linear-Aug-RN-SC (runs=25)
 - VICReg-Linear-Aug-RN-SC-RS (runs=25)
 - VICReg-Linear-Aug-RN-SC-RS-RM (runs=25)


,group,mean_test_f1,std_test_f1,n
0,VICReg-Linear-Aug-RN,0.426898,0.003895,25
1,VICReg-Linear-Aug-RN-SC,0.500595,0.005304,25
2,VICReg-Linear-Aug-RN-SC-RS,0.627527,0.006110,25
3,VICReg-Linear-Aug-RN-SC-RS-RM,0.663321,0.002880,25


,group_a,mean_a,n_a,group_b,mean_b,n_b,p_perm_one_sided_a_gt_b,p_ttest_two_sided,p_ttest_one_sided_a_gt_b
0,VICReg-Linear-Aug-RN,0.426898,25,VICReg-Linear-Aug-RN-SC,0.500595,25,1.0,1.402937e-42,1.0
1,VICReg-Linear-Aug-RN,0.426898,25,VICReg-Linear-Aug-RN-SC-RS,0.627527,25,1.0,4.347681e-56,1.0
2,VICReg-Linear-Aug-RN,0.426898,25,VICReg-Linear-Aug-RN-SC-RS-RM,0.663321,25,1.0,8.023284e-71,1.0
3,VICReg-Linear-Aug-RN-SC,0.500595,25,VICReg-Linear-Aug-RN-SC-RS,0.627527,25,1.0,1.513968e-51,1.0
4,VICReg-Linear-Aug-RN-SC,0.500595,25,VICReg-Linear-Aug-RN-SC-RS-RM,0.663321,25,1.0,1.955330e-51,1.0
5,VICReg-Linear-Aug-RN-SC-RS,0.627527,25,VICReg-Linear-Aug-RN-SC-RS-RM,0.663321,25,1.0,2.325515e-24,1.0



Saved:
 - paper_notebooks/csv/vicreg_groups_test_f1_summary.csv
 - paper_notebooks/csv/vicreg_groups_pairwise_pvals.csv


In [3]:
# VICREG p-value matrices across groups (t-test and permutation)
import pandas as pd
import numpy as np
import wandb
from pathlib import Path

# Stats
from scipy.stats import ttest_ind
try:
    from mlxtend.evaluate import permutation_test
except Exception:
    permutation_test = None

api = wandb.Api()
project_path = "francisco-soto-u-universidad-de-chile/Paper-Periodic-LightCurves-Classification"

# Fetch finished runs and keep only groups containing 'vicreg'
try:
    runs = api.runs(project_path, filters={"state": "finished"})
except Exception as e:
    print(f"W&B API error: {e}. Ensure WANDB_API_KEY is set and you have access.")
    runs = []

rows = []
for run in runs:
    grp = getattr(run, "group", None)
    if not isinstance(grp, str):
        continue
    grp_low = grp.lower()
    if "vicreg" not in grp_low:
        continue
    summary = run.summary._json_dict
    test_f1 = summary.get("test/f1")
    if isinstance(test_f1, (int, float)):
        rows.append({
            "run_name": run.name,
            "group": grp,
            "test_f1": float(test_f1),
        })

if not rows:
    print("No finished runs found with groups containing 'vicreg' and a numeric test/f1 in summary.")
else:
    df_v = pd.DataFrame(rows)
    vic_groups = sorted(df_v["group"].unique())
    print(f"Found VICREG groups (n={len(vic_groups)}):")
    for g in vic_groups:
        print(f" - {g} (runs={len(df_v[df_v['group']==g])})")
    if len(vic_groups) != 4:
        print("Warning: Expected 4 VICREG groups. Proceeding with available groups.")

    def get_arr(g):
        return df_v[df_v["group"] == g]["test_f1"].to_numpy()

    # Initialize symmetric matrices
    mat_t = pd.DataFrame(np.ones((len(vic_groups), len(vic_groups))), index=vic_groups, columns=vic_groups)
    mat_perm = pd.DataFrame(np.ones((len(vic_groups), len(vic_groups))), index=vic_groups, columns=vic_groups)

    for i, g1 in enumerate(vic_groups):
        arr1 = get_arr(g1)
        for j, g2 in enumerate(vic_groups):
            if j <= i:
                continue
            arr2 = get_arr(g2)
            # Welch t-test (two-sided)
            p_two = np.nan
            if len(arr1) > 1 and len(arr2) > 1:
                try:
                    tstat, p_two = ttest_ind(arr1, arr2, equal_var=False, nan_policy='omit')
                    p_two = float(p_two)
                except Exception:
                    p_two = np.nan
            mat_t.loc[g1, g2] = p_two
            mat_t.loc[g2, g1] = p_two

            # Permutation (two-sided approximation): 2*min(p(a>b), p(b>a)), clipped to 1
            p_perm_two = np.nan
            if permutation_test is not None and len(arr1) > 0 and len(arr2) > 0:
                try:
                    p_ab = float(permutation_test(arr1, arr2, func='x_mean > y_mean', method='approximate', num_rounds=10000, paired=False))
                    p_ba = float(permutation_test(arr2, arr1, func='x_mean > y_mean', method='approximate', num_rounds=10000, paired=False))
                    p_perm_two = min(1.0, 2 * min(p_ab, p_ba))
                except Exception:
                    p_perm_two = np.nan
            mat_perm.loc[g1, g2] = p_perm_two
            mat_perm.loc[g2, g1] = p_perm_two

    # Diagonal = 1.0 (no difference)
    for g in vic_groups:
        mat_t.loc[g, g] = 1.0
        mat_perm.loc[g, g] = 1.0

    print("\nWelch t-test (two-sided) p-value matrix:")
    display(mat_t)

    print("\nPermutation (two-sided approx) p-value matrix:")
    display(mat_perm)

    # Save CSVs
    out_dir = Path("paper_notebooks/csv")
    out_dir.mkdir(parents=True, exist_ok=True)
    mat_t.to_csv(out_dir / "vicreg_groups_pmatrix_ttest.csv")
    mat_perm.to_csv(out_dir / "vicreg_groups_pmatrix_permutation.csv")
    print("Saved matrices to:\n - paper_notebooks/csv/vicreg_groups_pmatrix_ttest.csv\n - paper_notebooks/csv/vicreg_groups_pmatrix_permutation.csv")


Found VICREG groups (n=4):
 - VICReg-Linear-Aug-RN (runs=25)
 - VICReg-Linear-Aug-RN-SC (runs=25)
 - VICReg-Linear-Aug-RN-SC-RS (runs=25)
 - VICReg-Linear-Aug-RN-SC-RS-RM (runs=25)

Welch t-test (two-sided) p-value matrix:


,VICReg-Linear-Aug-RN,VICReg-Linear-Aug-RN-SC,VICReg-Linear-Aug-RN-SC-RS,VICReg-Linear-Aug-RN-SC-RS-RM
VICReg-Linear-Aug-RN,1.000000e+00,1.402937e-42,4.347681e-56,8.023284e-71
VICReg-Linear-Aug-RN-SC,1.402937e-42,1.000000e+00,1.513968e-51,1.955330e-51
VICReg-Linear-Aug-RN-SC-RS,4.347681e-56,1.513968e-51,1.000000e+00,2.325515e-24
VICReg-Linear-Aug-RN-SC-RS-RM,8.023284e-71,1.955330e-51,2.325515e-24,1.000000e+00



Permutation (two-sided approx) p-value matrix:


,VICReg-Linear-Aug-RN,VICReg-Linear-Aug-RN-SC,VICReg-Linear-Aug-RN-SC-RS,VICReg-Linear-Aug-RN-SC-RS-RM
VICReg-Linear-Aug-RN,1.0000,0.0002,0.0002,0.0002
VICReg-Linear-Aug-RN-SC,0.0002,1.0000,0.0002,0.0002
VICReg-Linear-Aug-RN-SC-RS,0.0002,0.0002,1.0000,0.0002
VICReg-Linear-Aug-RN-SC-RS-RM,0.0002,0.0002,0.0002,1.0000


Saved matrices to:
 - paper_notebooks/csv/vicreg_groups_pmatrix_ttest.csv
 - paper_notebooks/csv/vicreg_groups_pmatrix_permutation.csv
